# 03 — Longitudinal Feature Engineering

**Inputs:** `Dataset_PlayerSeason_Modelling.csv`, `Role_Specific_Index_Dataset.csv`  
**Output:** `data/processed/Longitudinal_Features_Dataset.csv`

This notebook converts the audited player-season panel into season-`t` predictors for next-season forecasting. Lags are exact-season and gap-aware, so a missing season is not treated as an adjacent historical observation. The output contains **59 total columns**: 14 identifiers/context fields retained for traceability plus **45 predictor variables** grouped into current performance, trajectory/history, age/exposure, role, context/availability and role-conditional football output.

## 0. Setup

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.analysis_utils import paths, FEATURE_GROUPS, FEATURE_COLS
P = paths(ROOT)

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 30)

SEASON_ORDER = ['1718','1819','1920','2021','2122','2223','2324','2425']
SEASON_IDX = {s:i for i,s in enumerate(SEASON_ORDER)}


## 1. Load and merge

`Dataset_PlayerSeason_Modelling.csv` carries the raw per-90 statistics; `Role_Specific_Index_Dataset.csv` carries the role-specific performance index and its four raw components (`raw_attack`, `raw_creation`, `raw_possession`, `raw_defense`) built in the previous notebook. A `season_idx` integer (0=2017/18 ... 7=2024/25) makes lag arithmetic exact and gap-aware.

In [2]:
DATASET_DIR = P["processed"]

base = pd.read_csv(DATASET_DIR / "Dataset_PlayerSeason_Modelling.csv", dtype={'season': str})
idx = pd.read_csv(DATASET_DIR / "Role_Specific_Index_Dataset.csv", dtype={'season': str})

idx_cols = idx[['player_id','season','role','eligible_450','index_450min','index_450min_pctile',
                'raw_attack','raw_creation','raw_possession','raw_defense']]
df = base.merge(idx_cols, on=['player_id','season'], how='left')
df['season_idx'] = df['season'].map(SEASON_IDX)
df = df.sort_values(['player_id','season_idx']).reset_index(drop=True)
print("Merged:", df.shape)
df[['player','season','season_idx','role','index_450min']].head()


Merged: (21371, 195)


,player,season,season_idx,role,index_450min
0,Aaron Ciammaglichella,2425,7,MF,NaN
1,Aaron Connolly,1920,2,FW,-0.893590
2,Aaron Connolly,2021,3,FW,-0.414238
3,Aaron Connolly,2122,4,FW,NaN
4,Aaron Cresswell,1718,0,DF,-0.448491


## 2. Team-season strength

FBref's `Team Success_*` columns record team outcomes (goals for/against, points) **while the specific player was on the pitch**. Averaging these across a club's whole squad, weighted by minutes, approximates the club's actual season-level goal difference and points-per-match — without relying on any single player's on/off split.

`team_matches_available` (used later for the missed-match proxy) is the **90th percentile** of squad `Playing Time_MP`, not the max — a robustness choice after finding at least one raw-data row (Vitinha, PSG, 2019/20) whose `Playing Time_MP` was inflated by an upstream multi-competition scrape artefact; the 90th percentile is robust to that kind of single-row outlier while still reflecting a nearly-ever-present player's match count.

In [3]:
n90 = df['90s_'].replace(0, np.nan)
df['_team_gf90'] = df['Team Success_onG'] / n90
df['_team_ga90'] = df['Team Success_onGA'] / n90

def wavg(g, col, wcol='Playing Time_Min'):
    v, w = g[col], g[wcol].clip(lower=0)
    m = v.notna() & (w > 0)
    return np.average(v[m], weights=w[m]) if m.sum() > 0 else np.nan

team_rows = []
for (team, league, season), g in df.groupby(['team_primary','league_primary','season'], observed=True):
    team_rows.append({
        'team_primary': team, 'league_primary': league, 'season': season,
        'team_gf90': wavg(g, '_team_gf90'), 'team_ga90': wavg(g, '_team_ga90'),
        'team_ppm': wavg(g, 'Team Success_PPM'),
        'team_matches_available': g['Playing Time_MP'].quantile(0.9),
    })
team_strength = pd.DataFrame(team_rows)
team_strength['team_gd90'] = team_strength['team_gf90'] - team_strength['team_ga90']

print("Face validity check - strongest and weakest club-seasons by PPM:")
display(team_strength.sort_values('team_ppm', ascending=False).head(6)[['team_primary','league_primary','season','team_ppm']])
display(team_strength.sort_values('team_ppm').head(6)[['team_primary','league_primary','season','team_ppm']])


Face validity check - strongest and weakest club-seasons by PPM:


,team_primary,league_primary,season,team_ppm
400,Leverkusen,GER-Bundesliga,2324,2.652871
444,Manchester City,ENG-Premier League,1718,2.624397
412,Liverpool,ENG-Premier League,1920,2.602289
445,Manchester City,ENG-Premier League,1819,2.586761
411,Liverpool,ENG-Premier League,1819,2.559658
551,Paris S-G,FRA-Ligue 1,1920,2.527250


,team_primary,league_primary,season,team_ppm
660,Southampton,ENG-Premier League,2425,0.320135
653,Sheffield Utd,ENG-Premier League,2324,0.410648
326,Huddersfield,ENG-Premier League,1819,0.426929
624,Salernitana,ITA-Serie A,2324,0.465283
700,Toulouse,FRA-Ligue 1,1920,0.468144
18,Angers,FRA-Ligue 1,2223,0.484560


Elite sides (Man City, Liverpool, PSG, Bayern) top the list and relegation-zone sides (Southampton, Sheffield Utd, Huddersfield) sit at the bottom, in the expected seasons — this passes a basic face-validity check.

## 3. League-season strength (proxy — documented limitation)

There is no cross-league match data in this dataset (the five leagues never play each other), so a true Elo-style cross-league quality ranking cannot be estimated internally. `league_avg_ppm` is only a **within-league level control** (all leagues mechanically average close to the same PPM, since points are zero-sum within a league) — it is *not* a claim that, say, the Premier League is stronger than Ligue 1. A genuine cross-league strength ranking would need an external source (e.g. UEFA coefficients) and is flagged here as a limitation / future addition rather than silently approximated.

In [4]:
league_strength = team_strength.groupby(['league_primary','season'], observed=True).agg(
    league_avg_ppm=('team_ppm','mean')
).reset_index()

df = df.merge(team_strength[['team_primary','league_primary','season','team_ppm','team_gd90','team_gf90','team_ga90','team_matches_available']],
              on=['team_primary','league_primary','season'], how='left')
df = df.merge(league_strength, on=['league_primary','season'], how='left')
df['missed_matches_proxy'] = (df['team_matches_available'] - df['Playing Time_MP']).clip(lower=0)
print(league_strength.head())


       league_primary season  league_avg_ppm
0  ENG-Premier League   1718        1.379558
1  ENG-Premier League   1819        1.412233
2  ENG-Premier League   1920        1.385749
3  ENG-Premier League   2021        1.406276
4  ENG-Premier League   2122        1.394237


## 4. Pivot-based lag engine

For any column, a `player_id x season_idx` pivot table is built once; a lag-`k` lookup is then a simple merge on `season_idx - k`. Because missing seasons are simply absent rows/columns in the pivot, a genuine gap in a player's history (didn't meet the minutes threshold, was outside the top five leagues, etc.) correctly yields `NaN` rather than silently pulling the *previous available* season regardless of how long ago it was.

In [5]:
def pivot_lookup(col):
    return df.pivot_table(index='player_id', columns='season_idx', values=col, aggfunc='first')

def lag_series(col, k):
    piv = pivot_lookup(col)
    lag_df = df[['player_id','season_idx']].copy()
    lag_df['lookup_idx'] = lag_df['season_idx'] - k
    stacked = piv.stack(future_stack=True).rename(col).reset_index().rename(columns={'season_idx':'lookup_idx'})
    merged = lag_df.merge(stacked, on=['player_id','lookup_idx'], how='left')
    return merged[col].values

# --- validate on a known namesake + known transfer + known gap case before trusting it at scale ---
check_cols = ['player','season','season_idx','team_primary']
print("Namesake check (two different real 'Adama Traore's must not leak into each other's history):")
display(df[df['player']=='Adama Traoré'][check_cols + ['born_']])


Namesake check (two different real 'Adama Traore's must not leak into each other's history):


,player,season,season_idx,team_primary,born_
267,Adama Traoré,1718,0,Monaco,1995.0
268,Adama Traoré,1819,1,Monaco,1995.0
269,Adama Traoré,1920,2,Metz,1995.0
270,Adama Traoré,1819,1,Wolves,1996.0
271,Adama Traoré,1920,2,Wolves,1996.0
272,Adama Traoré,2021,3,Wolves,1996.0
273,Adama Traoré,2122,4,Wolves,1996.0
274,Adama Traoré,2223,5,Wolves,1996.0
275,Adama Traoré,2324,6,Fulham,1996.0
276,Adama Traoré,2425,7,Fulham,1996.0


The `player_id` (name+birth year) correctly keeps the Monaco/Metz Adama Traore (b.1995) and the Wolves/Barcelona/Fulham Adama Traore (b.1996) as separate histories — confirmed in the Dataset Audit notebook and re-verified here before it feeds any lag computation.

## 5. Performance rank, lags, and 1/2/3-season trends

In [6]:
df['perf_rank_t'] = df['index_450min_pctile']
df['perf_z_t'] = df['index_450min']
df['perf_z_t_minus1'] = lag_series('index_450min', 1)
df['perf_rank_t_minus1'] = lag_series('index_450min_pctile', 1)
df['perf_z_t_minus2'] = lag_series('index_450min', 2)
df['perf_z_t_minus3'] = lag_series('index_450min', 3)

# trend = average per-season change over the trailing 1/2/3-season window
df['trend_1s'] = df['perf_z_t'] - df['perf_z_t_minus1']
df['trend_2s'] = (df['perf_z_t'] - df['perf_z_t_minus2']) / 2
df['trend_3s'] = (df['perf_z_t'] - df['perf_z_t_minus3']) / 3

print("Coverage (non-null count):")
for c in ['perf_z_t_minus1','perf_z_t_minus2','perf_z_t_minus3','trend_1s','trend_2s','trend_3s']:
    print(f"  {c}: {df[c].notna().sum()} / {len(df)}  ({df[c].notna().mean()*100:.1f}%)")


Coverage (non-null count):
  perf_z_t_minus1: 10302 / 21371  (48.2%)
  perf_z_t_minus2: 7530 / 21371  (35.2%)
  perf_z_t_minus3: 5352 / 21371  (25.0%)
  trend_1s: 9127 / 21371  (42.7%)
  trend_2s: 6575 / 21371  (30.8%)
  trend_3s: 4632 / 21371  (21.7%)


Coverage falls with lag depth, as expected — fewer players have three prior *eligible* (>=450 min) seasons than one. This is a real constraint the modelling stage must handle (e.g. via a separate low-history model, or median-imputation with a missingness flag), not an error to fix here.

## 6. Age, minutes, starts, minutes volatility

In [7]:
df['age_sq'] = df['age_'] ** 2
df['minutes_t'] = df['Playing Time_Min']
df['starts_t'] = df['Playing Time_Starts']
df['minutes_t_minus1'] = lag_series('Playing Time_Min', 1)

min_t1 = lag_series('Playing Time_Min', 1)
min_t2 = lag_series('Playing Time_Min', 2)
mins_window = np.vstack([df['Playing Time_Min'].values, min_t1, min_t2]).T
df['minutes_volatility_3s'] = np.nanstd(mins_window, axis=1, ddof=0)
df.loc[np.isnan(mins_window).sum(axis=1) > 1, 'minutes_volatility_3s'] = np.nan  # need >=2 of 3 seasons

df[['player','season','age_','age_sq','minutes_t','minutes_volatility_3s']].sample(5, random_state=1)


,player,season,age_,age_sq,minutes_t,minutes_volatility_3s
20647,Yangel Herrera,2425,26.0,676.0,2257,577.8958
12980,Marek Hamšík,1718,30.0,900.0,2371,NaN
15443,Noni Madueke,2324,21.0,441.0,1053,205.0000
17861,Salvador Ichazo,1819,26.0,676.0,251,NaN
10926,Khanya Leshabela,2021,20.0,400.0,10,NaN


## 7. Role, and league/club strength level + change

In [8]:
df['is_DF'] = (df['role']=='DF').astype(int)
df['is_MF'] = (df['role']=='MF').astype(int)
df['is_FW'] = (df['role']=='FW').astype(int)

df['club_strength_t'] = df['team_ppm']
df['club_strength_t_minus1'] = lag_series('team_ppm', 1)
df['club_strength_change'] = df['club_strength_t'] - df['club_strength_t_minus1']
df['team_attack_strength_t'] = df['team_gf90']
df['team_defense_strength_t'] = df['team_ga90']
df['league_strength_t'] = df['league_avg_ppm']

df[['player','season','club_strength_t','club_strength_change','team_attack_strength_t','team_defense_strength_t']].sample(5, random_state=2)


,player,season,club_strength_t,club_strength_change,team_attack_strength_t,team_defense_strength_t
15027,Nemanja Radonjić,2223,1.358448,-0.510896,1.101883,1.095272
6248,Filipe Luís,1819,2.007571,-0.079319,1.450325,0.773595
11069,Koen Casteels,2324,1.092734,-0.354590,1.213893,1.635370
5791,Fabian Delph,1819,2.586761,-0.037636,2.500661,0.605306
8274,Iñaki Williams,2223,1.349741,-0.104577,1.234917,1.132400


## 8. Transfer and league-change indicators

Built the same gap-aware way: compares season-`t` club/league against the *exact* season-`t-1` value (not just "whatever row came before"), so a player re-entering the sample after an absence is correctly treated as unknown (`NaN`) rather than as a false transfer.

In [9]:
team_t1 = lag_series('team_primary', 1)
league_t1 = lag_series('league_primary', 1)
df['transfer_indicator'] = np.where(pd.isna(team_t1), np.nan, (df['team_primary'] != team_t1).astype(float))
df['league_change_indicator'] = np.where(pd.isna(league_t1), np.nan, (df['league_primary'] != league_t1).astype(float))
df['transferred_midseason_t'] = df['transferred_clubs'].astype(int)  # already-transferred-within-season flag, carried from the audit

print("transfer_indicator rate (of rows with a valid t-1):", round(df['transfer_indicator'].mean(),3))
print("league_change_indicator rate (of rows with a valid t-1):", round(df['league_change_indicator'].mean(),3))
print("(Cross-check: the Dataset Audit notebook found ~22-28% club change and ~9-13% league change among season-to-season retained players - matches.)")

print()
print("Validation - Aaron Ramsey has TWO real identities in this data (Arsenal/Juventus/Nice, b.1990 WAL; and a separate Burnley player, b.2003 ENG):")
display(df[df['player']=='Aaron Ramsey'][['player','season','season_idx','team_primary','transfer_indicator']])


transfer_indicator rate (of rows with a valid t-1): 0.262
league_change_indicator rate (of rows with a valid t-1): 0.105
(Cross-check: the Dataset Audit notebook found ~22-28% club change and ~9-13% league change among season-to-season retained players - matches.)

Validation - Aaron Ramsey has TWO real identities in this data (Arsenal/Juventus/Nice, b.1990 WAL; and a separate Burnley player, b.2003 ENG):


,player,season,season_idx,team_primary,transfer_indicator
35,Aaron Ramsey,1718,0,Arsenal,NaN
36,Aaron Ramsey,1819,1,Arsenal,0.0
37,Aaron Ramsey,1920,2,Juventus,1.0
38,Aaron Ramsey,2021,3,Juventus,0.0
39,Aaron Ramsey,2122,4,Juventus,0.0
40,Aaron Ramsey,2223,5,Nice,1.0
41,Aaron Ramsey,2324,6,Burnley,NaN


The 2023/24 Burnley row correctly shows `transfer_indicator = NaN`, not `1` — it belongs to a different real person (the age-2003 namesake) whose history starts there, not a continuation of the 1990-born player who was at Nice the season before.

## 9. Missed-match / injury proxy

In [10]:
df['missed_matches_t'] = df['missed_matches_proxy']
df['missed_matches_t_minus1'] = lag_series('missed_matches_proxy', 1)
df[['player','season','Playing Time_MP','team_matches_available','missed_matches_t']].sample(5, random_state=4)


,player,season,Playing Time_MP,team_matches_available,missed_matches_t
5621,Eric García,2223,24,36.0,12.0
4993,Dominic Solanke,2223,33,33.9,0.9
5870,Fabio Pisacane,2021,5,34.0,29.0
8200,Ivan Rakitić,1819,34,35.0,1.0
16771,Rani Khedira,2425,32,32.4,0.4


No genuine injury data exists in this dataset; `missed_matches_t` (team's ~90th-percentile match count minus the player's own appearances) is an availability proxy, not a clinical injury flag — it will also pick up rotation, suspension, and loan-adjacent squad churn, which is an acknowledged limitation.

## 10. Role-conditional attacking output

Per the brief: goals/xG/assists/xA should not be forced into the same interpretation across positions. Two representations are provided side by side:
- the **raw per-90 value**, present for every player (a defender's small goal tally is still informative, and tree-based models handle raw-value + role-dummy interactions natively);
- **role-interaction columns** (`stat_x_FW`, `stat_x_MF`, `stat_x_DF` — value where that role applies, 0 otherwise), so a linear/ridge baseline is not forced to fit one global coefficient for `goals_per90` across forwards and centre-backs alike.

In [11]:
df['goals_per90'] = df['Per 90 Minutes_Gls']
df['xG_per90'] = df['Per 90 Minutes_xG']
df['assists_per90'] = df['Per 90 Minutes_Ast']
df['xA_per90'] = df['Expected_xA'] / df['90s_'].replace(0, np.nan)

for stat in ['goals_per90','xG_per90','assists_per90','xA_per90']:
    df[f'{stat}_x_FW'] = df[stat] * df['is_FW']
    df[f'{stat}_x_MF'] = df[stat] * df['is_MF']
    df[f'{stat}_x_DF'] = df[stat] * df['is_DF']

# sanity check: role-interaction columns must be exactly 0 (not NaN) for the roles that don't apply
chk = df[(df['role']=='FW') & df['goals_per90'].notna()]
print("goals_per90_x_DF for forwards (must be 0):", chk['goals_per90_x_DF'].abs().sum())


goals_per90_x_DF for forwards (must be 0): 0.0


## 11. Progression, creation, possession, defensive metrics
Reuses the four role-aware per-90 components already built and validated in the index notebook, plus a standalone progression rate.

In [12]:
df['progression_per90'] = (df['Progression_PrgC'] + df['Progression_PrgP']) / df['90s_'].replace(0, np.nan)
df['creation_per90'] = df['raw_creation']
df['possession_per90'] = df['raw_possession']
df['defense_per90'] = df['raw_defense']


## 12. Number of available historical seasons
Count of the player's prior seasons in-sample (strictly before season `t`), regardless of contiguity — both a career-stage signal and a reliability flag for the trend features above (e.g. `trend_3s` is only meaningful when `n_hist_seasons >= 3`).

In [13]:
season_lists = df.groupby('player_id')['season_idx'].apply(lambda s: sorted(s.tolist()))
def n_hist(row):
    return sum(1 for s in season_lists[row['player_id']] if s < row['season_idx'])
df['n_hist_seasons'] = df.apply(n_hist, axis=1)

print(df[df['season_idx']==0]['n_hist_seasons'].describe())  # must be exactly 0 for every first-season row


count    2566.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: n_hist_seasons, dtype: float64


## 13. Assemble, verify, and save

In [14]:
KEEP = [
    'player_id','player','season','season_label','season_idx','team_primary','league_primary','role','age_','pos_',
    'perf_z_t','perf_rank_t',
    'perf_z_t_minus1','perf_rank_t_minus1','perf_z_t_minus2','perf_z_t_minus3',
    'trend_1s','trend_2s','trend_3s',
    'age_sq',
    'minutes_t','starts_t','minutes_t_minus1','minutes_volatility_3s',
    'is_DF','is_MF','is_FW',
    'league_strength_t','club_strength_t','club_strength_t_minus1','club_strength_change',
    'team_attack_strength_t','team_defense_strength_t',
    'goals_per90','xG_per90','assists_per90','xA_per90',
    'goals_per90_x_FW','goals_per90_x_MF','goals_per90_x_DF',
    'xG_per90_x_FW','xG_per90_x_MF','xG_per90_x_DF',
    'assists_per90_x_FW','assists_per90_x_MF','assists_per90_x_DF',
    'xA_per90_x_FW','xA_per90_x_MF','xA_per90_x_DF',
    'progression_per90','creation_per90','possession_per90','defense_per90',
    'missed_matches_t','missed_matches_t_minus1',
    'transfer_indicator','league_change_indicator','transferred_midseason_t',
    'n_hist_seasons',
]
assert all(c in df.columns for c in KEEP)
feat = df[KEEP].copy()
print("Final feature table:", feat.shape)
feat.head()


Final feature table: (21371, 59)


,player_id,player,season,season_label,season_idx,team_primary,league_primary,role,age_,pos_,perf_z_t,perf_rank_t,perf_z_t_minus1,perf_rank_t_minus1,perf_z_t_minus2,...,assists_per90_x_MF,assists_per90_x_DF,xA_per90_x_FW,xA_per90_x_MF,xA_per90_x_DF,progression_per90,creation_per90,possession_per90,defense_per90,missed_matches_t,missed_matches_t_minus1,transfer_indicator,league_change_indicator,transferred_midseason_t,n_hist_seasons
0,Aaron Ciammaglichella||2005,Aaron Ciammaglichella,2425,2024/25,7,Torino,ITA-Serie A,MF,19.0,MF,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32.1,NaN,NaN,NaN,0,0
1,Aaron Connolly||2000,Aaron Connolly,1920,2019/20,2,Brighton,ENG-Premier League,FW,19.0,FW,-0.893590,3.488372,NaN,NaN,NaN,...,0.0,0.000000,0.028571,0.0,0.000000,2.214286,3.464286,4.538018,1.642857,11.7,NaN,NaN,NaN,0,0
2,Aaron Connolly||2000,Aaron Connolly,2021,2020/21,3,Brighton,ENG-Premier League,FW,20.0,FW,-0.414238,27.069351,-0.893590,3.488372,NaN,...,0.0,0.000000,0.011364,0.0,0.000000,1.818182,3.295455,4.682060,1.420455,17.4,11.7,0.0,0.0,0,1
3,Aaron Connolly||2000,Aaron Connolly,2122,2021/22,4,Brighton,ENG-Premier League,FW,21.0,"FW,MF",NaN,NaN,-0.414238,27.069351,-0.89359,...,0.0,0.000000,0.058824,0.0,0.000000,2.352941,2.941176,3.684211,1.176471,30.0,17.4,0.0,0.0,0,2
4,Aaron Cresswell||1989,Aaron Cresswell,1718,2017/18,0,West Ham,ENG-Premier League,DF,27.0,DF,-0.448491,18.989071,NaN,NaN,NaN,...,0.0,0.087977,0.000000,0.0,0.070381,4.780059,5.322581,6.235841,5.454545,0.0,NaN,NaN,NaN,0,0


In [15]:
print("=== Missingness by feature (%) - sanity check, nothing should be unexpectedly high ===")
display((feat.isna().mean()*100).round(1).sort_values(ascending=False).to_frame('pct_missing'))


=== Missingness by feature (%) - sanity check, nothing should be unexpectedly high ===


,pct_missing
trend_3s,78.3
perf_z_t_minus3,75.0
trend_2s,69.2
perf_z_t_minus2,64.8
trend_1s,57.3
perf_z_t_minus1,51.8
perf_rank_t_minus1,51.8
minutes_t_minus1,39.6
league_change_indicator,39.6
club_strength_t_minus1,39.6


In [16]:
feat.to_csv(P["processed"] / "Longitudinal_Features_Dataset.csv", index=False)
print("Saved Longitudinal_Features_Dataset.csv:", feat.shape)


Saved Longitudinal_Features_Dataset.csv: (21371, 59)


## 14. Predictor dictionary and historical availability

For report traceability, the 45 canonical predictors are exported with their conceptual feature group and a concise definition. A separate availability table records how many player-season rows contain each predictor before model-stage imputation.

In [17]:
FEATURE_DESCRIPTIONS = {
'perf_rank_t':'Current season role-specific percentile rank (0–100).',
'perf_z_t':'Current season role-specific composite z-score.',
'perf_z_t_minus1':'Role-specific composite z-score from the exact previous season.',
'perf_z_t_minus2':'Role-specific composite z-score from exactly two seasons earlier (retained in the feature table but not in the canonical 45-feature model).',
'perf_z_t_minus3':'Role-specific composite z-score from exactly three seasons earlier (retained in the feature table but not in the canonical 45-feature model).',
'perf_rank_t_minus1':'Role-specific percentile rank from the exact previous season (retained for traceability but not in the canonical 45-feature model).',
'trend_1s':'Change in composite z-score from the exact previous season to season t.',
'trend_2s':'Average annualised change in composite z-score across a two-season span.',
'trend_3s':'Average annualised change in composite z-score across a three-season span.',
'n_hist_seasons':'Number of earlier player-seasons observed before season t.',
'age_':'Player age in season t.','age_sq':'Squared age term for nonlinear career-stage effects.',
'minutes_t':'Minutes played in season t.','starts_t':'Starts in season t.',
'minutes_t_minus1':'Minutes played in the exact previous season.',
'minutes_volatility_3s':'Variation in minutes across the available season-t/t-1/t-2 window, requiring at least two observations.',
'is_DF':'Defender role indicator.','is_MF':'Midfielder role indicator.','is_FW':'Forward role indicator.',
'league_strength_t':'Mean club points-per-match proxy within the player’s league and season.',
'club_strength_t':'Minutes-weighted club points-per-match proxy in season t.',
'club_strength_change':'Change in club-strength proxy relative to the exact previous season.',
'team_attack_strength_t':'Club goals-for-per-90 proxy in season t.',
'team_defense_strength_t':'Club goals-against-per-90 proxy in season t.',
'missed_matches_t':'Team-availability match-count proxy minus player appearances in season t.',
'missed_matches_t_minus1':'Exact previous-season value of the missed-matches availability proxy.',
'transfer_indicator':'Indicator that primary club changed from the exact previous season.',
'league_change_indicator':'Indicator that primary league changed from the exact previous season.',
'goals_per90':'Goals per 90 in season t.','xG_per90':'Expected goals per 90 in season t.',
'assists_per90':'Assists per 90 in season t.','xA_per90':'Expected assists per 90 in season t.',
'goals_per90_x_FW':'Goals-per-90 × forward indicator.','goals_per90_x_MF':'Goals-per-90 × midfielder indicator.','goals_per90_x_DF':'Goals-per-90 × defender indicator.',
'xG_per90_x_FW':'xG-per-90 × forward indicator.','xG_per90_x_MF':'xG-per-90 × midfielder indicator.','xG_per90_x_DF':'xG-per-90 × defender indicator.',
'assists_per90_x_FW':'Assists-per-90 × forward indicator.','assists_per90_x_MF':'Assists-per-90 × midfielder indicator.','assists_per90_x_DF':'Assists-per-90 × defender indicator.',
'xA_per90_x_FW':'xA-per-90 × forward indicator.','xA_per90_x_MF':'xA-per-90 × midfielder indicator.','xA_per90_x_DF':'xA-per-90 × defender indicator.',
'progression_per90':'Progressive carries plus progressive passes per 90.',
'creation_per90':'Creation component carried from the role-specific target construction.',
'possession_per90':'Possession component carried from the role-specific target construction.',
'defense_per90':'Defensive component carried from the role-specific target construction.',
}
feature_rows=[]
for group, cols in FEATURE_GROUPS.items():
    for col in cols:
        feature_rows.append({'feature':col,'feature_group':group,'definition':FEATURE_DESCRIPTIONS[col]})
feature_dictionary=pd.DataFrame(feature_rows)
assert len(feature_dictionary)==45 and set(feature_dictionary['feature'])==set(FEATURE_COLS)
feature_dictionary.to_csv(P['tables']/'03_feature_dictionary.csv',index=False)

availability_rows=[]
for col in FEATURE_COLS:
    availability_rows.append({'feature':col,'non_missing_n':int(feat[col].notna().sum()),'non_missing_pct':float(feat[col].notna().mean()*100)})
feature_availability=pd.DataFrame(availability_rows)
feature_availability.to_csv(P['tables']/'03_feature_availability_summary.csv',index=False)
print('Saved 45-feature dictionary and availability summary.')

Saved 45-feature dictionary and availability summary.


## Summary

The longitudinal table contains **21,371 player-season rows and 59 total columns**. Of those columns, **45 are canonical modelling predictors**; the remaining fields identify the player, season, role, team/league context and other traceability information.

Expected missingness is retained rather than filled at this stage:
- `perf_z_t`/`perf_rank_t` are unavailable for player-seasons outside the 450-minute outfield target population;
- lag and trend features are unavailable when the exact required prior season does not exist for the same `player_id`;
- rate statistics can be undefined where the underlying exposure denominator is zero.

Two contextual limitations are explicit. `league_strength_t` is a within-data league-level control rather than an externally calibrated league-quality rating, and `missed_matches_t` is an availability proxy that can reflect rotation, suspension or squad status as well as injury.

Notebook 04 adds the exact season-`t+1` target and evaluates RQ1–RQ3. Directional improve/stable/regress prediction is treated separately as the exploratory extension.

## Stage conclusion

The output preserves all 21,371 audited player-season rows. Historical missingness records the absence of an exact prior observation and is not filled using future information. Imputation is deferred to the modelling stage, where it is learned independently from the training partition of each chronological split.